# 👥 Phase 7: Customer Behavior & RFM Segmentation
**Project:** E-Commerce Sales Analytics Portfolio Project
**Dataset:** `data/cleaned/superstore_cleaned.csv`
**Objective:** Build a behavioral customer segmentation model using Recency, Frequency, and Monetary (RFM) analysis to evaluate customer lifetime value, engagement cohorts, and retention risk.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Load cleaned data
CLEANED_DATA_PATH = os.path.join('..', 'data', 'cleaned', 'superstore_cleaned.csv')
df = pd.read_csv(CLEANED_DATA_PATH, dtype={'postal_code': str})
df['order_date'] = pd.to_datetime(df['order_date'])

reference_date = df['order_date'].max()
print(f'RFM Reference Snapshot Date (Max Order Date): {reference_date.date()}')
print(f'Total Transaction Records: {len(df):,}')

## 1. Customer Base Aggregation & RFM Metric Derivation
- **Recency (R):** Number of calendar days between customer's latest order date and the snapshot reference date (`2017-12-30`).
- **Frequency (F):** Total number of distinct orders (`order_id`) placed by the customer.
- **Monetary (M):** Cumulative lifetime gross sales (`sales`) spend by the customer.

In [ ]:
rfm = df.groupby(['customer_id', 'customer_name', 'segment']).agg(
    recency=('order_date', lambda x: (reference_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('sales', 'sum'),
    total_profit=('profit', 'sum'),
    total_quantity=('quantity', 'sum'),
    first_order_date=('order_date', 'min'),
    last_order_date=('order_date', 'max')
).reset_index()

rfm['aov'] = (rfm['monetary'] / rfm['frequency']).round(2)
rfm['profit_margin_pct'] = ((rfm['total_profit'] / rfm['monetary']) * 100).round(2)

print(f'Unique Customer Base: {len(rfm)} accounts')
rfm.head(5)

## 2. Quantile Scoring Methodology (1 to 5 Scores)
- **Recency Score (1-5):** Segmented into quintiles where **5 represents the most recent purchasers** (lowest days) and **1 represents the least recent**.
- **Frequency Score (1-5):** Quantile scored on order counts (using `rank(method='first')` to handle tied frequencies) where **5 is highest frequency**.
- **Monetary Score (1-5):** Quantile scored on total spend where **5 is highest spend**.

In [ ]:
rfm['r_score'] = pd.qcut(rfm['recency'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)
rfm['rfm_combined_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

# Segment Assignment Function
def assign_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2 and m >= 2:
        return 'Potential Loyalists'
    elif r >= 4 and f == 1 and m == 1:
        return 'New Customers'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r >= 3 and f <= 2 and m <= 2:
        return 'Need Attention'
    elif r <= 2 and f <= 2 and m >= 3:
        return 'About to Sleep'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost Customers'
    else:
        return 'Hibernating'

rfm['rfm_segment'] = rfm.apply(assign_segment, axis=1)

# Export for Power BI and Analytics
os.makedirs(os.path.join('..', 'data', 'analytics'), exist_ok=True)
rfm.to_csv(os.path.join('..', 'data', 'analytics', 'customer_rfm.csv'), index=False)
print('Customer RFM dataset exported successfully.')

## 3. Segment-Level Performance Analysis
Quantifying customer counts, revenue shares, net profits, and average metrics across behavioral segments.

In [ ]:
seg_summary = rfm.groupby('rfm_segment').agg(
    customer_count=('customer_id', 'count'),
    total_sales=('monetary', 'sum'),
    total_profit=('total_profit', 'sum'),
    avg_sales=('monetary', 'mean'),
    avg_profit=('total_profit', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_recency=('recency', 'mean')
).reset_index()

seg_summary['pct_customers'] = (seg_summary['customer_count'] / len(rfm)) * 100
seg_summary['pct_sales'] = (seg_summary['total_sales'] / rfm['monetary'].sum()) * 100
seg_summary['pct_profit'] = (seg_summary['total_profit'] / rfm['total_profit'].sum()) * 100
seg_summary['profit_margin_pct'] = (seg_summary['total_profit'] / seg_summary['total_sales']) * 100

seg_summary.sort_values('total_sales', ascending=False)

## 4. Reconciliation & Data Validation
Validating that customer metrics perfectly sum to baseline cleaned dataset totals.

In [ ]:
print('=== Data Reconciliation Checks ===')
print(f'Customer Count: {len(rfm)} (Expected: 793) -> Match: {len(rfm) == 793}')
print(f'Total Sales: ${rfm["monetary"].sum():,.2f} (Expected: $2,297,200.86) -> Match: {abs(rfm["monetary"].sum() - 2297200.86) < 0.01}')
print(f'Total Profit: ${rfm["total_profit"].sum():,.2f} (Expected: $286,397.02) -> Match: {abs(rfm["total_profit"].sum() - 286397.02) < 0.01}')
print(f'Duplicate Customer Records: {rfm["customer_id"].duplicated().sum()}')
print(f'Unassigned Segments: {rfm["rfm_segment"].isnull().sum()}')